In [8]:
from sklearn.ensemble import RandomForestRegressor
import os
import numpy as np
import pandas as pd
import sys
sys.path.append(os.path.dirname(os.getcwd()))
import functions

Data wrangling for the input data to models

In [9]:
df = pd.read_parquet('../data/features_table.parquet')

# add an order_number to each row in df, in order of the order_date
df['order_number'] = df['order_date'].rank(method='first').astype(int)

# add a delivery_date column to df, which is the order_date plus the actual_days
df['delivery_date'] = df['order_date'] + pd.to_timedelta(df['actual_days'], unit='D')

# Do one-hot encoding now, so that the train and test columns match
features_data = functions.one_hot_encode(df, ['origin_country', 'us_destination_state'])

In [10]:
# Choose the run_date for the train-test split
run_date = features_data['order_date'].quantile(0.9)
print(run_date)
# The window of dates will be one week
print(run_date + pd.Timedelta(days=7))

2017-12-26 00:00:00
2018-01-02 00:00:00


Random Forest Model
* Machine learning model that utilizes decision trees
* Original formulation for one date, for testing during development

In [11]:
# Get data
X_train, y_train, X_test, y_test, feature_cols = functions.train_test_split(
    features_data, run_date, window_days=7)

In [12]:
# Simple Random Forest regressor
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [13]:
# Testing data dimension columns
rf_data = (
    df.loc[
        (df["order_date"] >= run_date)
        & (df["order_date"] <= run_date + pd.Timedelta(days=7)),
        ["origin_country", "us_destination_state", "order_date", "order_number"],
    ]
    .reset_index(drop=True)
    .copy()
)
rf_data

,origin_country,us_destination_state,order_date,order_number
0,Vietnam,PR,2017-12-26,681
1,Vietnam,PR,2017-12-26,682
2,Vietnam,CA,2017-12-26,683
3,Vietnam,PR,2017-12-27,684
4,Vietnam,PR,2017-12-27,685
5,Vietnam,OH,2017-12-28,686
6,Vietnam,MD,2017-12-28,687
7,Vietnam,OH,2017-12-28,688
8,Vietnam,OH,2017-12-29,689
9,Vietnam,MO,2017-12-29,690


In [14]:
rf_results, rf_mape = functions.summarize_results(rf_data,
                                                  run_date,
                                                  y_test,
                                                  y_pred,
                                                  model_name='Random Forest',
                                                  window_days=7)
print('Unique Origin-Destination pairs:',len(rf_results[['origin_country', 'us_destination_state']].drop_duplicates()))
rf_results

Predicted rows: 19
 MAE: 1.043 
 RMSE: 1.257 
 R2: 0.000 
 MAPE: 0.307
Unique Origin-Destination pairs: 9


,origin_country,us_destination_state,order_date,order_number,actual,pred,abs_err,model
0,Vietnam,PR,2017-12-26,681,4,4.18,0.18,Random Forest
1,Vietnam,PR,2017-12-26,682,3,4.18,1.18,Random Forest
2,Vietnam,CA,2017-12-26,683,2,3.87,1.87,Random Forest
3,Vietnam,PR,2017-12-27,684,4,4.18,0.18,Random Forest
4,Vietnam,PR,2017-12-27,685,3,4.61,1.61,Random Forest
5,Vietnam,OH,2017-12-28,686,6,4.80,1.20,Random Forest
6,Vietnam,MD,2017-12-28,687,4,2.93,1.07,Random Forest
7,Vietnam,OH,2017-12-28,688,4,3.56,0.44,Random Forest
8,Vietnam,OH,2017-12-29,689,5,4.80,0.20,Random Forest
9,Vietnam,MO,2017-12-29,690,5,6.00,1.00,Random Forest


In [15]:
# How much weight did the model put on each feature?
feature_importances = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)

# How many observations were in the training set for each feature?
feature_counts = pd.DataFrame({'feature': feature_cols, 'count': X_train[feature_cols].sum()}).sort_values('count', ascending=False)

# Merge feature importance and counts to see if there's a relationship between them
feature_analysis = pd.merge(feature_importances, feature_counts, on='feature')
feature_analysis.sort_values('importance', ascending=False)

,feature,importance,count
0,projected_days,0.676386,2117
1,us_destination_state_FL,0.029902,25
2,us_destination_state_IL,0.026057,42
3,us_destination_state_MO,0.023831,6
4,us_destination_state_TX,0.023339,36
5,us_destination_state_SC,0.020695,6
6,us_destination_state_DC,0.019879,9
7,us_destination_state_MD,0.019823,8
8,us_destination_state_PR,0.017408,234
9,us_destination_state_MI,0.017067,22


Evaluation across multiple time periods (folds) for the Random Forest Model

In [16]:
# Choose the run_date for the train-test split
run_date = features_data['order_date'].quantile(0.5)
print(run_date)
# The window of dates will be one week
print(run_date + pd.Timedelta(days=7))

# Create a list of run_dates to loop through
run_dates = pd.date_range(start=run_date, end=features_data["order_date"].max() - pd.Timedelta(days=14), freq='7D')
run_dates

2016-01-28 00:00:00
2016-02-04 00:00:00


DatetimeIndex(['2016-01-28', '2016-02-04', '2016-02-11', '2016-02-18',
               '2016-02-25', '2016-03-03', '2016-03-10', '2016-03-17',
               '2016-03-24', '2016-03-31',
               ...
               '2017-11-16', '2017-11-23', '2017-11-30', '2017-12-07',
               '2017-12-14', '2017-12-21', '2017-12-28', '2018-01-04',
               '2018-01-11', '2018-01-18'],
              dtype='datetime64[us]', length=104, freq='7D')

In [17]:
mape_records = []
for r in run_dates:
    # Get data
    X_train, y_train, X_test, y_test, feature_cols = functions.train_test_split(features_data, 
                                                                                r,
                                                                                window_days=7)
    # Only run the model if there are rows in the test set for the given run_date
    if(len(X_test) > 0):
        # Testing data dimension columns
        rf_data = (
            df.loc[
                (df["order_date"] >= r)
                & (df["order_date"] <= r + pd.Timedelta(days=7)),
                ["origin_country", "us_destination_state", "order_date", "order_number"],
            ]
            .reset_index(drop=True)
            .copy()
        )
        # Run the model
        model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        # Get the predicted values for the test set
        y_pred = model.predict(X_test)
        results, mape = functions.summarize_results(rf_data,
                                                          r,
                                                          y_test,
                                                          y_pred, 
                                                          model_name='Random Forest',
                                                          print_stats=False,
                                                          window_days=7)
        mape_records.append({'run_date': r, 'mape': round(mape, 3)})

rf_mape_df = pd.DataFrame(mape_records)
rf_mape_df

,run_date,mape
0,2016-01-28,2.470000e-01
1,2016-02-04,1.820000e-01
2,2016-02-11,7.330000e-01
3,2016-02-18,3.780000e-01
4,2016-02-25,2.730000e-01
5,2016-03-03,2.110000e-01
6,2016-03-10,2.280000e-01
7,2016-03-17,4.250000e-01
8,2017-11-09,5.920000e-01
9,2017-11-16,3.630000e-01


In [18]:
# remove outliers from rf_mape_df and average the rest
q_high = rf_mape_df['mape'].quantile(0.99)
rf_mape_df_filtered = rf_mape_df[(rf_mape_df['mape'] <= q_high)]
rf_mape_avg = rf_mape_df_filtered['mape'].mean()
rf_mape_avg

np.float64(0.34650000000000003)

In [19]:
display(rf_mape_df_filtered)

,run_date,mape
0,2016-01-28,0.247
1,2016-02-04,0.182
2,2016-02-11,0.733
3,2016-02-18,0.378
4,2016-02-25,0.273
5,2016-03-03,0.211
6,2016-03-10,0.228
7,2016-03-17,0.425
8,2017-11-09,0.592
9,2017-11-16,0.363


In [20]:
# However, the if the lead times are longer than, for example, 30 days, 
#   then the Delivery Date will be a few weeks ahead of the Order Date 
# Run Date (Now)
# Order Date (Next Week),
# Delivery Date (e.g. more than 30 days ahead) 
# This means that the model accuracy would need to be tracked by the order_number
#   And this model accuracy + tracking would not be complete until the delivery date has passed for all orders in the test set

# Next step: Improve the evaluation method to account for this scenario